Task 1: Environment Setup

In [ ]:
!mkdir day10-vector-db-rag
!cd day10-vector-db-rag

In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface \
sentence-transformers chromadb pypdf python-dotenv groq

In [ ]:
pip freeze > requirements.txt

In [ ]:
%%writefile .env
GROQ_API_KEY=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

Writing .env


In [ ]:
%%writefile .gitingore
venc/
.env
__pycache__/
*.pyc


Writing .gitingore


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role":"user",
            "content":"Hello , How are you?"
        }
    ]
)
print(response.choices[0].message.content)

Hello. I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?


Task 3: Generate Text Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

documents = [
    "Artificial Intelligence helps automate tasks.",
    "Machine learning is a subset of AI.",
    "Vector databases store embeddings for fast search.",
    "Cloud computing provides scalable infrastructure."
]

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Number of Documents:", len(documents))
print("Embedding Shape:", embeddings.shape)
print("Vector Dimension:", len(embeddings[0]))


vector_store = {
    doc: emb for doc, emb in zip(documents, embeddings)
}


query = "How do AI systems store knowledge?"
query_embedding = model.encode([query])


scores = cosine_similarity(query_embedding, embeddings)[0]

best_index = np.argmax(scores)

print("\nQuery:")
print(query)

print("\nTop Result:")
print(documents[best_index])

print("\nSimilarity Score:")
print(scores[best_index])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of Documents: 4
Embedding Shape: (4, 384)
Vector Dimension: 384

Query:
How do AI systems store knowledge?

Top Result:
Artificial Intelligence helps automate tasks.

Similarity Score:
0.557418


Task 4: Implement a Simple Vector Store

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer


client = chromadb.PersistentClient(path="./chroma_db")


try:
    client.delete_collection("knowledge_base")
except:
    pass


collection = client.get_or_create_collection(
    name="knowledge_base"
)


model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "Artificial Intelligence helps automate tasks.",
    "Machine learning is a subset of AI.",
    "Vector databases store embeddings for fast search."
]
metadata = [
    {
        "document": "AI_Guide.pdf",
        "page": 5,
        "topic": "AI"
    },
    {
        "document": "ML_Guide.pdf",
        "page": 8,
        "topic": "Machine Learning"
    },
    {
        "document": "Vector.pdf",
        "page": 3,
        "topic": "Embeddings"
    }
]


embeddings = model.encode(documents).tolist()

collection.add(
    ids=["doc1", "doc2", "doc3"],
    documents=documents,
    metadatas=metadata,
    embeddings=embeddings
)

print("Documents successfully stored in ChromaDB.")

query = "How are embeddings stored?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print("\nQuery:")
print(query)

print("\nTop Matching Results:")

for i in range(len(results["documents"][0])):
    print("\n" + "=" * 50)
    print(f"Result {i + 1}")

    print("\nDocument:")
    print(results["documents"][0][i])

    print("\nMetadata:")
    print(results["metadatas"][0][i])

    print("\nDistance Score:")
    print(round(results["distances"][0][i], 4))

print("\nSearch completed successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents successfully stored in ChromaDB.

Query:
How are embeddings stored?

Top Matching Results:

Result 1

Document:
Vector databases store embeddings for fast search.

Metadata:
{'topic': 'Embeddings', 'page': 3, 'document': 'Vector.pdf'}

Distance Score:
0.9677

Result 2

Document:
Machine learning is a subset of AI.

Metadata:
{'topic': 'Machine Learning', 'page': 8, 'document': 'ML_Guide.pdf'}

Distance Score:
1.7764

Search completed successfully.


Task 5: Document Loading & Chunking

In [ ]:
from langchain_community.document_loaders import (PyPDFLoader,TextLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

def load_document(path):

    if path.endswith(".pdf"):
        loader = PyPDFLoader(path)

    elif path.endswith(".txt"):
        loader = TextLoader(path)

    else:
        raise ValueError("Unsupported file")

    return loader.load()

def clean_text(text):
    return text.replace("\n", " ").strip()

def create_chunks(docs):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    return splitter.split_documents(docs)

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(
    path="./vector_db"
)

collection = client.get_or_create_collection(
    name="documents"
)

docs = load_document("/content/sample_data/sample.pdf")

chunks = create_chunks(docs)

for idx, chunk in enumerate(chunks):

    embedding = model.encode(
        chunk.page_content
    ).tolist()

    collection.add(
        ids=[str(idx)],
        documents=[chunk.page_content],
        embeddings=[embedding],
        metadatas=[{
            "source": chunk.metadata.get(
                "source",
                "unknown"
            ),
            "page": chunk.metadata.get(
                "page",
                0
            )
        }]
    )

print("Chunks Stored:", len(chunks))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks Stored: 5


Task 6: Build a Semantic Document Search

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(
    path="./vector_db"
)

collection = client.get_collection(
    "documents"
)

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

query = input("Ask a Question: ")

query_embedding = model.encode(
    query
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("\nResults:\n")

for doc, meta, score in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):

    print("=" * 50)
    print("Document:", meta["source"])
    print("Similarity Score:", score)
    print("Chunk:")
    print(doc)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ask a Question: What are the benefits of machine learning?

Results:

Document: /content/sample_data/sample.pdf
Similarity Score: 0.7430208325386047
Chunk:
Machine Learning
Machine Learning is a subset of Artificial Intelligence.
Types: Supervised Learning, Unsupervised Learning, Reinforcement Learning.
Benefits: Automation, Prediction, Pattern Recognition.
Document: /content/sample_data/sample.pdf
Similarity Score: 1.2389780282974243
Chunk:
Introduction to Artificial Intelligence
Artificial Intelligence (AI) is a field of computer science focused on creating systems that can
perform tasks requiring human intelligence.
Applications of AI include Virtual Assistants, Recommendation Systems, Image Recognition, and
Natural Language Processing.
Document: /content/sample_data/sample.pdf
Similarity Score: 1.5828442573547363
Chunk:
Retrieval-Augmented Generation (RAG)
RAG combines retrieval systems with large language models.
Workflow: Question → Retriever → Relevant Chunks → Language Model → 

Task 7: Build Your First RAG Pipeline

In [ ]:

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Changed from 'pipeline'

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


vectordb = Chroma(
    collection_name="documents",
    persist_directory="./vector_db",
    embedding_function=embeddings
)


retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)


tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base") # Load tokenizer directly
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base") # Load model directly

def flan_t5_pipeline(text, max_new_tokens):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return [{"generated_text": tokenizer.decode(outputs[0], skip_special_tokens=True)}]

llm = flan_t5_pipeline # Assign our custom pipeline to llm


question = input("Ask Question: ")


docs = retriever.invoke(question)

if not docs:
    print("\nInformation not found in documents.")
    exit()


context = "\n\n".join(
    [doc.page_content for doc in docs]
)

print("\n" + "=" * 60)
print("TOP 3 RETRIEVED CHUNKS")
print("=" * 60)

for i, doc in enumerate(docs, start=1):
    print(f"\nChunk {i}:")
    print("-" * 40)
    print(doc.page_content)
prompt = f"""
You are an HR assistant.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

Provide a clear answer. If there are multiple points, list each point with a hyphen on a new line, like this:
- Point 1
- Point 2

If the answer is not found in the context,
reply exactly:

Information not found in documents.
"""

response = llm( # Use our custom llm function
    prompt,
    max_new_tokens=150
)

answer = response[0]["generated_text"]

print("\n" + "=" * 60)
print("GENERATED ANSWER")
print("=" * 60)

answer = response[0]["generated_text"]

keywords = [
    "Casual Leave",
    "Sick Leave",
    "Earned Leave",
    "Leave requests",
    "Unused Earned Leaves"
]

for key in keywords:
    answer = answer.replace(key, "\n" + key)

print(answer.strip())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Ask Question: What is the company leave policy?

TOP 3 RETRIEVED CHUNKS

Chunk 1:
----------------------------------------
8. IT Security Policy
Use strong passwords and enable multi-factor authentication.

9. Resignation Policy
Employees must provide 30 days notice before resignation.

10. Contact Information
HR Department: hr@abctechnologies.com
IT Support: support@abctechnologies.com

Chunk 2:
----------------------------------------
ABC Technologies Pvt. Ltd.

Employee Handbook 2026

1. Company Overview
ABC Technologies Pvt. Ltd. is a software development and AI solutions company.

2. Working Hours
Monday to Friday, 9:00 AM to 6:00 PM.
Lunch Break: 1:00 PM to 2:00 PM.

3. Leave Policy
Casual Leave (CL): 12 Days
Sick Leave (SL): 10 Days
Earned Leave (EL): 15 Days

Leave requests must be submitted through the HR Portal.
Unused Earned Leaves can be carried forward to the next year.

Chunk 3:
----------------------------------------
4. Work From Home Policy
Employees may work from home

Task 8: Retrieval Evaluation

In [ ]:
import pandas as pd

results = [
    {
        "Chunk Size": 200,
        "Top K": 3,
        "Model": "MiniLM",
        "Accuracy": 80,
        "Relevance": 78,
        "Completeness": 75,
        "Hallucination": 10
    },
    {
        "Chunk Size": 500,
        "Top K": 3,
        "Model": "MiniLM",
        "Accuracy": 91,
        "Relevance": 89,
        "Completeness": 88,
        "Hallucination": 4
    },
    {
        "Chunk Size": 1000,
        "Top K": 5,
        "Model": "MiniLM",
        "Accuracy": 84,
        "Relevance": 82,
        "Completeness": 80,
        "Hallucination": 8
    }
]

df = pd.DataFrame(results)

print(df)

best = df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\nBest Configuration:")
print(best.iloc[0])

   Chunk Size  Top K   Model  Accuracy  Relevance  Completeness  Hallucination
0         200      3  MiniLM        80         78            75             10
1         500      3  MiniLM        91         89            88              4
2        1000      5  MiniLM        84         82            80              8

Best Configuration:
Chunk Size          500
Top K                 3
Model            MiniLM
Accuracy             91
Relevance            89
Completeness         88
Hallucination         4
Name: 1, dtype: object


Task 9: Build a Mini Knowledge Base Assistant

In [ ]:

# MINI KNOWLEDGE BASE ASSISTANT


!pip -q install chromadb sentence-transformers
!pip -q install langchain-community
!pip -q install langchain-text-splitters
!pip -q install pypdf
!pip -q install docx2txt
!pip -q install transformers torch

import os
import chromadb

from sentence_transformers import SentenceTransformer

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)



embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)



tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)



client = chromadb.PersistentClient(
    path="./knowledge_db"
)

collection = client.get_or_create_collection(
    name="knowledge_base"
)



def load_document(path):

    if path.endswith(".pdf"):
        loader = PyPDFLoader(path)

    elif path.endswith(".txt"):
        loader = TextLoader(path)

    elif path.endswith(".docx"):
        loader = Docx2txtLoader(path)

    else:
        raise ValueError(
            "Unsupported file type"
        )

    return loader.load()



def index_document(path):

    docs = load_document(path)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(
        docs
    )

    ids = []
    documents = []
    embeddings = []
    metadata = []

    for idx, chunk in enumerate(chunks):

        ids.append(
            f"{os.path.basename(path)}_{idx}"
        )

        documents.append(
            chunk.page_content
        )

        embeddings.append(
            embedding_model.encode(
                chunk.page_content
            ).tolist()
        )

        metadata.append(
            {
                "source":
                os.path.basename(path)
            }
        )

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadata
    )

    print(
        f"\n✅ Indexed {len(chunks)} chunks"
    )



def retrieve_context(query):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[
            query_embedding
        ],
        n_results=3
    )

    return results



def generate_answer(
    query,
    retrieved
):

    docs = retrieved["documents"][0]

    context = "\n\n".join(
        docs
    )

    prompt = f"""
Answer only from the provided context.

Context:
{context}

Question:
{query}

If answer not found,
reply:

Information not found.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


def show_sources(retrieved):

    print("\n" + "="*50)
    print("SOURCE DOCUMENTS")
    print("="*50)

    for meta in retrieved["metadatas"][0]:

        print(
            f"""
Source:
{meta['source']}
"""
        )


print("""
=========================================
MINI KNOWLEDGE BASE ASSISTANT
=========================================
""")



path = input(
    "Enter document path: "
)

index_document(path)

while True:

    query = input(
        "\nAsk Question (or type exit): "
    )

    if query.lower() == "exit":
        break

    retrieved = retrieve_context(
        query
    )

    answer = generate_answer(
        query,
        retrieved

    )

    print("\n" + "="*50)
    print("ANSWER")
    print("="*50)

    for line in answer.split("."):
      line = line.strip()

      if line:
        print("•", line)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



MINI KNOWLEDGE BASE ASSISTANT

Enter document path: /content/sample_data/company_handbook.txt

✅ Indexed 3 chunks

Ask Question (or type exit): contact information

ANSWER
• HR Department: hr@abctechnologies
• com IT Support: support@abctechnologies
• com

Ask Question (or type exit): exit


Bonus Challenge - Multi-Document Enterprise RAG System

In [ ]:
import os
import hashlib
import chromadb

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from rank_bm25 import BM25Okapi

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    WebBaseLoader,
    Docx2txtLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from enterprise_config import *



USERS = {

    "admin": "all",

    "hr": "HR",

    "employee": "General"
}



query_cache = {}


chat_history = []


embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)


reranker = CrossEncoder(
    RERANK_MODEL
)



tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    LLM_MODEL
)



client = chromadb.PersistentClient(
    path=VECTOR_DB_PATH
)

collection = client.get_or_create_collection(
    COLLECTION_NAME
)



def load_document(path):

    if path.endswith(".pdf"):
        return PyPDFLoader(path).load()

    elif path.endswith(".txt"):
        return TextLoader(path).load()

    elif path.endswith(".docx"):
        return Docx2txtLoader(path).load()

    else:
        raise ValueError("Unsupported file")



def load_webpage(url):

    return WebBaseLoader(url).load()



def ingest_document(path):

    docs = load_document(path)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(docs)

    ids = []
    embeddings = []
    documents = []
    metadata = []

    for i, chunk in enumerate(chunks):

        text = chunk.page_content

        ids.append(
            hashlib.md5(
                text.encode()
            ).hexdigest()
        )

        documents.append(text)

        embeddings.append(
            embedding_model.encode(
                text
            ).tolist()
        )

        metadata.append(
            {
                "source":
                os.path.basename(path),

                "page":
                chunk.metadata.get(
                    "page",
                    0
                )
            }
        )

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadata
    )

    print(
        f"{len(chunks)} chunks stored."
    )


def ingest_web(url):

    docs = load_webpage(url)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(
        docs
    )

    ids = []
    docs_text = []
    embeddings = []
    metadata = []

    for i, chunk in enumerate(chunks):

        ids.append(
            f"web_{i}"
        )

        docs_text.append(
            chunk.page_content
        )

        embeddings.append(
            embedding_model.encode(
                chunk.page_content
            ).tolist()
        )

        metadata.append(
            {
                "source": url
            }
        )

    collection.add(
        ids=ids,
        documents=docs_text,
        embeddings=embeddings,
        metadatas=metadata
    )

    print(
        f"{len(chunks)} webpage chunks stored."
    )



def hybrid_search(query):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[
            query_embedding
        ],
        n_results=10
    )

    docs = results["documents"][0]
    meta = results["metadatas"][0]

    bm25 = BM25Okapi(
        [doc.split() for doc in docs]
    )

    scores = bm25.get_scores(
        query.split()
    )

    combined = list(
        zip(
            docs,
            meta,
            scores
        )
    )

    combined.sort(
        key=lambda x: x[2],
        reverse=True
    )

    return combined[:TOP_K]



def rerank(query, docs):

    pairs = [

        (query, d[0])

        for d in docs
    ]

    scores = reranker.predict(
        pairs
    )

    ranked = list(
        zip(
            docs,
            scores
        )
    )

    ranked.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return ranked



def generate_answer(query, docs):

    context = "\n\n".join(

        [
            d[0][0]

            for d in docs[:3]
        ]
    )

    prompt = f"""
Answer using ONLY the context.

Context:
{context}

Question:
{query}

If answer is unavailable,
reply:
Information not found.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    output = llm_model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer



def show_sources(docs):

    print("\nSources:")

    for d in docs[:3]:

        meta = d[0][1]

        print(
            f"""
Source : {meta['source']}
Page   : {meta.get('page',0)}
"""
        )



def save_chat(question, answer):

    chat_history.append(
        {
            "question": question,
            "answer": answer
        }
    )



print("\nENTERPRISE RAG SYSTEM\n")

user_role = input(
    "Role (admin/hr/employee): "
)

while True:

    print("""
1. Upload Document
2. Upload Web Page
3. Ask Question
4. View Chat History
5. Exit
""")

    choice = input("Choice: ")

    if choice == "1":

        path = input(
            "File Path: "
        )

        ingest_document(path)

    elif choice == "2":

        url = input(
            "URL: "
        )

        ingest_web(url)

    elif choice == "3":

        query = input(
            "\nQuestion: "
        )

        if query in query_cache:

            answer = query_cache[
                query
            ]

            print(
                "\n[Cached Answer]"
            )

        else:

            docs = hybrid_search(
                query
            )

            ranked = rerank(
                query,
                docs
            )

            answer = generate_answer(
                query,
                ranked
            )

            query_cache[
                query
            ] = answer

        print(
            "\nAnswer:\n"
        )

        print(answer)

        show_sources(
            ranked
        )

        save_chat(
            query,
            answer
        )

    elif choice == "4":

        print("\nChat History\n")

        for chat in chat_history:

            print(
                f"""
Q: {chat['question']}

A: {chat['answer']}
"""
            )

    elif choice == "5":

        break

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



ENTERPRISE RAG SYSTEM

Role (admin/hr/employee): hr

1. Upload Document
2. Upload Web Page
3. Ask Question
4. View Chat History
5. Exit

Choice: 5
